# NAVDRIFT-0 · ISRO SIH 2026 PS #26168
## AI Speed Estimator — IO-VNBD LSTM Training
**Model**: LSTM (speed from IMU) + CNN-1D (vibration filter)  
**Dataset**: IO-VNBD Smartphone stream (S-*), 10 Hz, 24 columns  
**Target**: `speed` (m/s) from [ax, ay, az, gx, gy, gz, gravity_x, gravity_y, gravity_z]  
**Runtime**: A100 GPU · checkpoints saved to Google Drive

## ─── CELL 1 · Anti-disconnect (run first, keep this cell alive) ───

In [ ]:
# Prevents Colab from timing out. Clicks the page every 60 s.
# Run this cell ONCE and leave it — it runs in the browser, not Python.
from IPython.display import display, Javascript
display(Javascript('''
function clickConnect() {
  console.log("[NAVDRIFT] Anti-disconnect ping");
  // Click the connect button if disconnected
  var btn = document.querySelector("colab-connect-button");
  if (btn) { btn.click(); }
  // Also click the "Run anyway" button on dialogs
  var dialogs = document.querySelectorAll('.dialogButton');
  dialogs.forEach(d => { if(d.textContent.includes('Run anyway')) d.click(); });
}
setInterval(clickConnect, 60000);
console.log("[NAVDRIFT] Anti-disconnect active — pings every 60s");
'''))
print("✓ Anti-disconnect active in browser (JS running every 60s)")

## ─── CELL 2 · Mount Google Drive + create checkpoint dir ───

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/NAVDRIFT0'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
DATA_DIR   = f'{DRIVE_ROOT}/IO-VNBD'
EXPORT_DIR = f'{DRIVE_ROOT}/exports'

for d in [CKPT_DIR, DATA_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'✓ Drive mounted')
print(f'  Checkpoints → {CKPT_DIR}')
print(f'  Dataset     → {DATA_DIR}')
print(f'  Exports     → {EXPORT_DIR}')

## ─── CELL 3 · Install deps + imports ───

In [ ]:
!pip install -q onnx onnxruntime tf2onnx

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import json, os, time, zipfile, urllib.request

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TF {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

## ─── CELL 4 · Download IO-VNBD from GitHub ───

In [ ]:
# IO-VNBD GitHub: https://github.com/onyekpeu/IO-VNBD
# Smartphone (S-*) zip — ~150 MB
ZIP_URL  = 'https://github.com/onyekpeu/IO-VNBD/raw/master/Synchronised%20V%20and%20S%20Dataset.zip'
ZIP_PATH = f'{DATA_DIR}/IO-VNBD-sync.zip'
EXTRACT  = f'{DATA_DIR}/extracted'

if not os.path.exists(ZIP_PATH):
    print('Downloading IO-VNBD (this may take a minute)…')
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print(f'✓ Downloaded → {ZIP_PATH}')
else:
    print(f'✓ Already downloaded: {ZIP_PATH}')

if not os.path.exists(EXTRACT):
    os.makedirs(EXTRACT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT)
    print(f'✓ Extracted to {EXTRACT}')

# List available files
all_files = sorted(Path(EXTRACT).rglob('*.csv'))
s_files = [f for f in all_files if f.stem.startswith('S-')]
v_files = [f for f in all_files if f.stem.startswith('V-')]
print(f'\nSmartphone files (S-*): {len(s_files)}')
for f in s_files[:5]: print(f'  {f.name}')
print(f'Vehicle ECU files (V-*): {len(v_files)}')

## ─── CELL 5 · Load + inspect dataset ───

In [ ]:
# Load all smartphone CSVs
dfs = []
for f in s_files:
    try:
        df = pd.read_csv(f, low_memory=False)
        df['source_file'] = f.stem
        dfs.append(df)
    except Exception as e:
        print(f'  SKIP {f.name}: {e}')

raw = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(raw):,}   Columns: {raw.columns.tolist()}')
raw.head(3)

In [ ]:
# ── Column mapping (IO-VNBD smartphone stream) ──
# Detect actual column names flexibly
cols = raw.columns.str.lower().tolist()
print('All columns:', raw.columns.tolist())

def find_col(candidates, df):
    for c in candidates:
        matches = [col for col in df.columns if c.lower() in col.lower()]
        if matches: return matches[0]
    return None

COL = {
    'ax':    find_col(['accel_x','accelerometerx','acc_x','linear_x'], raw),
    'ay':    find_col(['accel_y','accelerometery','acc_y','linear_y'], raw),
    'az':    find_col(['accel_z','accelerometerz','acc_z','linear_z'], raw),
    'gx':    find_col(['gyro_x','gyrox','yaw_rate','gyroscope_x'], raw),
    'gy':    find_col(['gyro_y','gyroy','pitch_rate','gyroscope_y'], raw),
    'gz':    find_col(['gyro_z','gyroz','roll_rate','gyroscope_z'], raw),
    'grav_x':find_col(['gravity_x','gravx'], raw),
    'grav_y':find_col(['gravity_y','gravy'], raw),
    'grav_z':find_col(['gravity_z','gravz'], raw),
    'speed': find_col(['speed','velocity','gps_speed'], raw),
    'lat':   find_col(['lat','latitude'], raw),
    'lon':   find_col(['lon','lng','longitude'], raw),
    'heading':find_col(['heading','bearing','orientation','yaw'], raw),
    'ts':    find_col(['timestamp','time_ms','millis'], raw),
}
print('\nColumn mapping:')
for k,v in COL.items(): print(f'  {k:10} → {v}')

## ─── CELL 6 · Preprocessing + feature engineering ───

In [ ]:
# Build feature matrix — handle missing gravity columns gracefully
FEAT_COLS_RAW = ['ax','ay','az','gx','gy','gz']
if COL['grav_x']: FEAT_COLS_RAW += ['grav_x','grav_y','grav_z']

feat_map  = {k: COL[k] for k in FEAT_COLS_RAW if COL[k]}
speed_col = COL['speed']
lat_col   = COL['lat']
lon_col   = COL['lon']

needed = list(feat_map.values()) + [speed_col, lat_col, lon_col]
needed = [c for c in needed if c is not None]

data = raw[needed].copy()
data.columns = [k for k,v in feat_map.items()] + ['speed','lat','lon'][:len(needed)-len(feat_map)]

# Coerce numeric, drop NaN
data = data.apply(pd.to_numeric, errors='coerce').dropna()

# Derived features
data['accel_mag']  = np.sqrt(data['ax']**2 + data['ay']**2 + data['az']**2)
data['gyro_mag']   = np.sqrt(data['gx']**2 + data['gy']**2 + data['gz']**2)

# Rolling stats (window=5 → 0.5 s at 10 Hz)
for c in ['ax','ay','az','gx','gy','gz']:
    data[f'{c}_std5'] = data[c].rolling(5, min_periods=1).std().fillna(0)

# Remove stationary noise: speed < 0.2 m/s → clamp to 0
data['speed'] = data['speed'].clip(lower=0)
data.loc[data['speed'] < 0.2, 'speed'] = 0.0

FEATURE_COLS = [c for c in data.columns if c not in ['speed','lat','lon']]
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Samples: {len(data):,}  |  Speed range: {data["speed"].min():.2f}–{data["speed"].max():.2f} m/s')
data[FEATURE_COLS + ['speed']].describe()

## ─── CELL 7 · Build sliding-window sequences ───

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

SEQ_LEN  = 50   # 5 seconds at 10 Hz
STRIDE   = 5    # 0.5 s stride

X_raw = data[FEATURE_COLS].values.astype(np.float32)
y_raw = data['speed'].values.astype(np.float32)

# Normalise features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Save scaler params to Drive
np.save(f'{EXPORT_DIR}/scaler_mean.npy', scaler.mean_)
np.save(f'{EXPORT_DIR}/scaler_scale.npy', scaler.scale_)
np.save(f'{EXPORT_DIR}/feature_cols.npy', np.array(FEATURE_COLS))

# Sliding window
Xs, ys = [], []
for i in range(0, len(X_scaled) - SEQ_LEN, STRIDE):
    Xs.append(X_scaled[i:i+SEQ_LEN])
    ys.append(y_raw[i+SEQ_LEN-1])   # predict speed at END of window

Xs = np.array(Xs, dtype=np.float32)
ys = np.array(ys, dtype=np.float32)

# Train / val / test split (70/15/15)
n = len(Xs)
n_tr = int(n*0.70); n_val = int(n*0.85)
X_tr, X_val, X_te = Xs[:n_tr], Xs[n_tr:n_val], Xs[n_val:]
y_tr, y_val, y_te = ys[:n_tr], ys[n_tr:n_val], ys[n_val:]

print(f'Sequences: train={len(X_tr):,}  val={len(X_val):,}  test={len(X_te):,}')
print(f'Shape: {X_tr.shape}  →  target: ({y_tr.shape},)')

## ─── CELL 8 · Model definition — LSTM + CNN ensemble ───

In [ ]:
N_FEAT = len(FEATURE_COLS)

# ── Model A: Bidirectional LSTM ──────────────────────────────────────────────
def build_lstm(seq_len=SEQ_LEN, n_feat=N_FEAT):
    inp = keras.Input(shape=(seq_len, n_feat), name='imu_seq')
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inp)
    x = layers.Dropout(0.25)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, activation='relu', name='speed_mps')(x)  # speed ≥ 0
    return keras.Model(inp, out, name='NAVDRIFT_LSTM')

# ── Model B: 1D-CNN (faster, for TFLite export) ──────────────────────────────
def build_cnn1d(seq_len=SEQ_LEN, n_feat=N_FEAT):
    inp = keras.Input(shape=(seq_len, n_feat), name='imu_seq')
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(inp)
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.20)(x)
    out = layers.Dense(1, activation='relu', name='speed_mps')(x)
    return keras.Model(inp, out, name='NAVDRIFT_CNN1D')

lstm_model  = build_lstm()
cnn_model   = build_cnn1d()

for m in [lstm_model, cnn_model]:
    m.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='huber',              # robust to outlier speed spikes
        metrics=['mae']
    )
    m.summary()
    print()

## ─── CELL 9 · Train LSTM (primary model) ───

In [ ]:
EPOCHS    = 60
BATCH     = 256
CKPT_PATH = f'{CKPT_DIR}/lstm_best.keras'

callbacks = [
    keras.callbacks.ModelCheckpoint(
        CKPT_PATH, monitor='val_mae', save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_mae', patience=10, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.CSVLogger(f'{CKPT_DIR}/lstm_training_log.csv', append=True),
]

# Resume from checkpoint if exists
if os.path.exists(CKPT_PATH):
    print(f'Resuming from {CKPT_PATH}')
    lstm_model = keras.models.load_model(CKPT_PATH)

history_lstm = lstm_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH,
    callbacks=callbacks,
    verbose=1
)
print('✓ LSTM training complete. Best saved to Drive.')

## ─── CELL 10 · Train CNN-1D (TFLite export model) ───

In [ ]:
CKPT_CNN = f'{CKPT_DIR}/cnn1d_best.keras'

cbs_cnn = [
    keras.callbacks.ModelCheckpoint(CKPT_CNN, monitor='val_mae', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_mae', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    keras.callbacks.CSVLogger(f'{CKPT_DIR}/cnn_training_log.csv', append=True),
]

if os.path.exists(CKPT_CNN):
    print(f'Resuming CNN from {CKPT_CNN}')
    cnn_model = keras.models.load_model(CKPT_CNN)

history_cnn = cnn_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=40, batch_size=512,
    callbacks=cbs_cnn, verbose=1
)
print('✓ CNN-1D training complete.')

## ─── CELL 11 · Evaluate + plot training curves ───

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_plt, hist, name in [
    (axes[0], history_lstm, 'LSTM'),
    (axes[1], history_cnn,  'CNN-1D')
]:
    ax_plt.plot(hist.history['mae'],     label='Train MAE')
    ax_plt.plot(hist.history['val_mae'], label='Val MAE')
    ax_plt.set_title(f'{name} — Speed MAE (m/s)')
    ax_plt.set_xlabel('Epoch'); ax_plt.set_ylabel('MAE (m/s)')
    ax_plt.legend(); ax_plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{EXPORT_DIR}/training_curves.png', dpi=150)
plt.show()

# Test set evaluation
for model, name in [(lstm_model,'LSTM'),(cnn_model,'CNN-1D')]:
    loss, mae = model.evaluate(X_te, y_te, verbose=0)
    print(f'{name}: test MAE = {mae:.3f} m/s  ({mae*3.6:.2f} km/h)')

## ─── CELL 12 · Position plot from test set (ATE benchmark) ───

In [ ]:
# Reconstruct trajectory from test-set predictions
# Use LSTM predicted speed + gyro heading integration

test_start = n_val * STRIDE          # row index in original data
test_data  = data.iloc[test_start + SEQ_LEN : test_start + SEQ_LEN + len(y_te)].copy()
test_data  = test_data.reset_index(drop=True)

DT = 0.1  # 10 Hz
pred_speed = lstm_model.predict(X_te, batch_size=512, verbose=0).flatten()
pred_speed = np.maximum(pred_speed, 0)   # enforce ≥ 0

# Heading from gyro gz integration (if available, else use GPS heading)
gz_col = feat_map.get('gz')
use_gz = gz_col is not None and gz_col in test_data.columns

# Initialize at first valid GPS fix
lat0 = test_data['lat'].iloc[0]
lon0 = test_data['lon'].iloc[0]

pred_lat = [lat0]; pred_lon = [lon0]
gt_lat   = test_data['lat'].tolist()
gt_lon   = test_data['lon'].tolist()

heading = 0.0
for i in range(1, len(pred_speed)):
    if use_gz:
        gz_val = test_data.iloc[i].get(gz_col, 0) if gz_col in test_data.columns else 0
        heading += gz_val * DT
    else:
        # Approximate from consecutive GPS
        dlat = gt_lat[i] - gt_lat[i-1]
        dlon = gt_lon[i] - gt_lon[i-1]
        if abs(dlat)+abs(dlon) > 1e-7:
            heading = np.arctan2(dlon, dlat)

    spd = pred_speed[i]
    d_lat = spd * np.cos(heading) * DT / 111000
    d_lon = spd * np.sin(heading) * DT / (111000 * np.cos(np.radians(pred_lat[-1])))
    pred_lat.append(pred_lat[-1] + d_lat)
    pred_lon.append(pred_lon[-1] + d_lon)

pred_lat = np.array(pred_lat); pred_lon = np.array(pred_lon)
gt_lat_a = np.array(gt_lat);   gt_lon_a = np.array(gt_lon)

# ATE (Average Trajectory Error) in metres
def haversine_m(la1, lo1, la2, lo2):
    R = 6371000
    dlat = np.radians(la2-la1); dlon = np.radians(lo2-lo1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(la1))*np.cos(np.radians(la2))*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

ate_per_pt = haversine_m(pred_lat, pred_lon, gt_lat_a[:len(pred_lat)], gt_lon_a[:len(pred_lat)])
ATE   = np.mean(ate_per_pt)
ATE_max = np.max(ate_per_pt)

# Total distance travelled
total_dist = np.sum(haversine_m(gt_lat_a[:-1], gt_lon_a[:-1], gt_lat_a[1:], gt_lon_a[1:]))
drift_pct  = (ATE / total_dist * 100) if total_dist > 0 else 0

print(f'\n═══ NAVDRIFT-0 BENCHMARK (IO-VNBD test set) ═══')
print(f'  ATE (mean)    : {ATE:.2f} m')
print(f'  ATE (max)     : {ATE_max:.2f} m')
print(f'  Total distance: {total_dist:.0f} m')
print(f'  Drift %       : {drift_pct:.2f}%  (target: <10%)')
if drift_pct < 10:
    print(f'  ✓ PASSES PS benchmark (<10% drift)')
else:
    print(f'  ✗ FAILS benchmark — tune model or use map matching')

# ── Position Plot ──
fig, ax = plt.subplots(figsize=(10, 8))
n_plot = min(len(pred_lat), len(gt_lat_a))
ax.plot(gt_lon_a[:n_plot],  gt_lat_a[:n_plot],  'b-', lw=1.5, label='Ground Truth (GPS)', alpha=0.8)
ax.plot(pred_lon[:n_plot],  pred_lat[:n_plot],   'r-', lw=1.5, label=f'NAVDRIFT-0 LSTM (ATE={ATE:.1f}m)', alpha=0.8)
ax.plot(gt_lon_a[0], gt_lat_a[0],  'go', ms=10, label='Start')
ax.plot(gt_lon_a[n_plot-1], gt_lat_a[n_plot-1], 'rs', ms=10, label='End')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'NAVDRIFT-0 · IO-VNBD Test Trajectory\nATE={ATE:.2f}m | Drift={drift_pct:.2f}% | Total={total_dist:.0f}m')
ax.legend(loc='best'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{EXPORT_DIR}/position_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✓ Position plot saved → {EXPORT_DIR}/position_plot.png')

## ─── CELL 13 · Export TFLite (on-device inference) ───

In [ ]:
# Export CNN-1D as TFLite (smaller, faster for phones)
converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]    # int8 quantization

def representative_data_gen():
    for i in range(0, min(500, len(X_tr)), 10):
        yield [X_tr[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
tflite_path  = f'{EXPORT_DIR}/navdrift_speed.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'✓ TFLite model: {len(tflite_model)/1024:.1f} KB → {tflite_path}')

# Also save LSTM as SavedModel for edge engine (ONNX)
lstm_save = f'{CKPT_DIR}/lstm_savedmodel'
lstm_model.save(lstm_save)
print(f'✓ LSTM SavedModel → {lstm_save}')

# Save metadata
meta = {
    'seq_len': SEQ_LEN,
    'feature_cols': FEATURE_COLS,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'lstm_test_mae_ms': float(lstm_model.evaluate(X_te, y_te, verbose=0)[1]),
    'cnn_test_mae_ms':  float(cnn_model.evaluate(X_te, y_te, verbose=0)[1]),
    'ate_mean_m': float(ATE),
    'drift_pct': float(drift_pct),
    'total_dist_m': float(total_dist),
}
with open(f'{EXPORT_DIR}/model_meta.json','w') as f:
    json.dump(meta, f, indent=2)
print(f'✓ Metadata saved → {EXPORT_DIR}/model_meta.json')

## ─── CELL 14 · Export ONNX (for edge Python engine) ───

In [ ]:
import tf2onnx, onnx

# Convert LSTM to ONNX for navdrift_engine.py
input_sig = [tf.TensorSpec([1, SEQ_LEN, N_FEAT], tf.float32, name='imu_seq')]
onnx_model_proto, _ = tf2onnx.convert.from_keras(lstm_model, input_signature=input_sig)
onnx_path = f'{EXPORT_DIR}/navdrift_lstm.onnx'
with open(onnx_path, 'wb') as f:
    f.write(onnx_model_proto.SerializeToString())
print(f'✓ ONNX model → {onnx_path}')

# Quick ONNX runtime test
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
dummy = X_te[:1]
out = sess.run(None, {'imu_seq': dummy})
tf_out = lstm_model.predict(dummy, verbose=0)
print(f'  ONNX output: {out[0][0][0]:.4f} m/s  |  TF output: {tf_out[0][0]:.4f} m/s  — match: {np.allclose(out[0], tf_out, atol=0.01)}')

## ─── CELL 15 · Summary + what's in Drive ───
```
MyDrive/NAVDRIFT0/
├── checkpoints/
│   ├── lstm_best.keras         ← best LSTM checkpoint
│   ├── cnn1d_best.keras        ← best CNN-1D checkpoint
│   ├── lstm_savedmodel/        ← TF SavedModel
│   ├── lstm_training_log.csv
│   └── cnn_training_log.csv
└── exports/
    ├── navdrift_speed.tflite   ← on-device (Android/iOS)
    ├── navdrift_lstm.onnx      ← edge Python engine
    ├── position_plot.png       ← SIH proposal figure
    ├── training_curves.png
    ├── model_meta.json         ← ATE, drift%, MAE
    ├── scaler_mean.npy
    ├── scaler_scale.npy
    └── feature_cols.npy
```

In [ ]:
print('═'*55)
print('  NAVDRIFT-0 Training Complete')
print('═'*55)
print(f'  LSTM test MAE : {meta["lstm_test_mae_ms"]:.3f} m/s')
print(f'  CNN  test MAE : {meta["cnn_test_mae_ms"]:.3f} m/s')
print(f'  ATE (mean)    : {meta["ate_mean_m"]:.2f} m')
print(f'  Drift         : {meta["drift_pct"]:.2f}%  (benchmark: <10%)')
print(f'  TFLite model  : {EXPORT_DIR}/navdrift_speed.tflite')
print(f'  ONNX model    : {EXPORT_DIR}/navdrift_lstm.onnx')
print(f'  Position plot : {EXPORT_DIR}/position_plot.png')
print('═'*55)